In [24]:
import os
os.chdir('/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/')

# general
import glob
import datetime as dt

# data 
import xarray as xr 
import numpy as np
import pandas as pd
from utils.helper_funcs import calculate_wind_components

# plotting
import matplotlib.pyplot as plt


In [25]:
def produce_df_from_ds(ds, precip_benchmark, precip_test):
    # Convert to pandas for event grouping
    df = pd.DataFrame({
        precip_benchmark: ds[precip_benchmark].to_series(),
        precip_test: ds[precip_test].to_series(),
    })
    return df

def find_events_union(ds, precip_benchmark, precip_test, precip_thresh=0.0, max_gap_steps=1, resample_interval=None, limit=50, site='unknown', to_xarray=False, min_event_length=0):
    """
    Find precipitation events defined by the union of all columns in df:
      - an index belongs to an event if ANY column > precip_thresh
      - small dry gaps (<= max_gap_steps) between wet samples are filled/merged
    Returns a DataFrame with event_id, start_idx, end_idx, start_time (optional),
    end_time (optional), per-sensor sums, diff and abs_diff (benchmark minus others).
    """
    df = produce_df_from_ds(ds, precip_benchmark, precip_test)
    if resample_interval:
        df = df.resample(resample_interval).sum()
    else:
        resample_interval = '30min'
    
    if 'min' in resample_interval:
        resample_hours = int(resample_interval.replace('min','')) / 60
    elif 'h' in resample_interval:
        resample_hours = int(resample_interval.replace('h',''))
    elif 'd' in resample_interval:
        resample_hours = int(resample_interval.replace('d',''))
    else:
        print("Resample interval not recognized:", resample_interval)
        return
    
    # boolean wet series: True if any sensor reports > threshold at that time
    wet = (df > precip_thresh).any(axis=1).to_numpy(dtype=bool)
    true_idx = np.where(wet)[0] # indices of True values
    if len(true_idx) == 0:
        return pd.DataFrame()  # no events

    # fill/merge small dry gaps between True indices
    wet_filled = wet.copy()
    gaps = np.diff(true_idx) - 1  # number of dry steps between consecutive trues
    for i, gap in enumerate(gaps):
        if 0 < gap <= max_gap_steps: # if the gap is small enough, fill it
            start = true_idx[i] + 1 # fill it with the true index value
            end = true_idx[i + 1] # at the end, set to next true index
            wet_filled[start:end] = True # do this until this is no longer true

    # find start/end indices of contiguous True runs in wet_filled
    N = len(wet_filled)
    # starts where False -> True
    # locate where previous value is dry and the next is wet, then the next index is a new event
    starts = np.where((~wet_filled[:-1]) & (wet_filled[1:]))[0] + 1 if N > 1 else ( [0] if wet_filled[0] else [] )
    if wet_filled[0]:
        starts = np.concatenate(([0], starts)) if len(starts) > 0 else np.array([0])
    # ends where True -> False
    # locate where previous value is wet and the next is dry, then the current index is the end of an event
    ends = np.where((wet_filled[:-1]) & (~wet_filled[1:]))[0] if N > 1 else ( [0] if wet_filled[0] else [] )
    if wet_filled[-1]:
        ends = np.concatenate((ends, [N - 1])) if len(ends) > 0 else np.array([N - 1])

    starts = np.asarray(starts, dtype=int)
    ends = np.asarray(ends, dtype=int)

    # Build results
    rows = []
    # assume first column is benchmark (sensor A)
    benchmark = df.columns[0]
    for eid, (s, e) in enumerate(zip(starts, ends), start=1):
        block = df.iloc[s:e + 1]
        sums = block.sum()
        # compute difference relative to benchmark (benchmark - each other)
        # here we compute a single scalar diff as benchmark_total - mean(other totals)
        # but we'll also provide per-sensor totals so you can choose how to measure bias.
        benchmark_total = sums[benchmark]
        # choose a simple measure: benchmark minus mean of all others
        others = sums.drop(benchmark)
        if len(others) == 0:
            diff = 0.0
        else:
            diff = benchmark_total - others.mean()
        row = {
            "event_id": eid,
            "event_duration": (e - s) * resample_hours, # in hours
            "start_time": df.index[s] if isinstance(df.index, pd.DatetimeIndex) else None,
            "end_time": df.index[e] if isinstance(df.index, pd.DatetimeIndex) else None,
            "benchmark_total": float(benchmark_total),
            "others_total_mean": float(others.mean()) if len(others) else np.nan,
            "diff": float(diff),
            "abs_diff": abs(float(diff))
        }
        # add each sensor total as separate keys
        # for col in df.columns:
        #     row[f"total_{col}"] = float(sums[col])
        rows.append(row)
    # create the DataFrame
    df_out = pd.DataFrame(rows)

    # drop rows with event_durations less than min_event_length
    df_out = df_out[df_out['event_duration'] > min_event_length]

    # make event_id the index
    df_out = df_out.set_index('event_id')

    # sort by abs_diff descending
    df_out = df_out.sort_values(by='abs_diff', ascending=False)

    # reset the index to reset the event_id numbering and add 1 to start from 1
    df_out = df_out.reset_index()
    df_out['event_id'] = df_out.index + 1
    df_out = df_out.set_index('event_id')
    
    # limit to top N events
    if limit:
        df_out = df_out.iloc[0:limit]

    if to_xarray:
        return events_to_xarray(df_out, test_gauge=precip_test, benchmark_gauge=precip_benchmark,
                                precip_threshold=precip_thresh,
                                dry_gap_hours=max_gap_steps * resample_hours,
                                site=site)
    else:
        return df_out # limit to top 50 events for brevity


def events_to_xarray(events_df, test_gauge, benchmark_gauge, precip_threshold, dry_gap_hours, site):
    events_ds = events_df.to_xarray()

    # add a dimension for the sensor
    events_ds = events_ds.expand_dims({'test_instrument': [test_gauge],
                                    'benchmark':[benchmark_gauge]})
    # create name for attribute assignment
    benchmark_name = benchmark_gauge.replace('_',' ').upper()
    test_gauge_name = test_gauge.replace('_',' ').upper()

    # add attrubutes
    variable_attributes_dict = {
        'event_duration': {'units': 'hrs',
                        'longname':'Duration of the event in hours'},
        'start_time': {'longname':'Start time of the event'},
        'end_time': {'longname':'End time of the event'},
        'benchmark_total': {'units':'mm',
                            'longname':f'Total precipitation from benchmark gauge ({benchmark_name}) over the event'},
        'others_total_mean': {'units':'mm',
                            'longname':f'Mean total precipitation from {test_gauge_name} over the event'},
        'diff': {'units':'mm',
                    'longname':f'Difference in total precipitation (benchmark minus {test_gauge_name}) over the event'},
        'abs_diff': {'units':'mm',
                        'longname':f'Absolute difference in total precipitation (benchmark minus {test_gauge_name}) over the event'},
    }

    global_attributes_dict = {
        'title': f'Precipitation Event Comparison between benchmarks and gauges',
        'site': site,
        'date_created': dt.datetime.now().isoformat(),
        'precip_threshold_mm': precip_threshold,
        'max_dry_gap_hours': dry_gap_hours,
        'description': f'This dataset contains precipitation event statistics comparing different benchmarks and gauges at the {site} site.',
    }

    dimension_atrribute_dict = {
        'event_id': {'longname':'Unique identifier for each precipitation event'},
        'test_instrument': {'longname':'Name of the test precipitation gauge/instrument'},
        'benchmark': {'longname':'Name of the benchmark precipitation gauge/instrument'},
    }

    for var in events_ds.data_vars:
        if var in variable_attributes_dict:
            events_ds[var].attrs.update(variable_attributes_dict[var])
    for dim in events_ds.dims:
        if dim in dimension_atrribute_dict:
            events_ds[dim].attrs.update(dimension_atrribute_dict[dim])
    events_ds.attrs.update(global_attributes_dict)
    return events_ds



In [26]:
# Parameters
PRECIP_THRESHOLD = 0.05     # mm — to filter noise
DRY_GAP_STEPS = int(12 / 0.5) # number of 30-min intervals, 6-hours in this case

OUTPUT_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis/'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [27]:
def fast_mode_rounded(x):
    arr = x.to_numpy()
    arr = arr[~np.isnan(arr)]  # drop NaNs fast
    if arr.size == 0:
        return np.nan
    rounded = (10 * np.round(arr / 10)).astype(int)
    vals, counts = np.unique(rounded, return_counts=True)
    return vals[np.argmax(counts)]
def event_stats(met_var, time, start, end):
    """Compute event-averaged stats for each event window."""
    results = []
    for s, e in zip(start, end):
        sel = met_var.sel(time=slice(s, e))
        try: 
            results.append([
            sel.mean().item(),
            sel.max().item(),
            sel.min().item(),
            sel.std().item()
        ])
        except:
            results.append([np.nan, np.nan, np.nan, np.nan])
        
    return np.array(results)

In [107]:
def combine_events(event_dataset, met_dataset, vars_to_add):
    """Add meteorological statistics to each event in the event dataset."""
    benchmark_ds_list = []   

    for benchmark in event_dataset.benchmark.values:
        new_ds_list = []  # Reset inside benchmark loop

        for test_gauge in event_dataset.test_instrument.values:
            if test_gauge == benchmark:
                continue  # skip self comparisons

            ds = event_dataset.sel(benchmark=benchmark, test_instrument=test_gauge)

            for var in vars_to_add:
                if var != 'wdir_vec_mean':
                    stats = event_stats(
                        met_var=met_dataset[var],
                        time=met_dataset['time'],
                        start=ds['start_time'].values,
                        end=ds['end_time'].values
                    )
                    ds[f'{var}_mean'] = (('event_id'), stats[:,0])
                    ds[f'{var}_max'] = (('event_id'), stats[:,1])
                    ds[f'{var}_min'] = (('event_id'), stats[:,2])
                    ds[f'{var}_std'] = (('event_id'), stats[:,3])
                else:
                    if 'wdir_vec_mean' not in met_dataset:
                        continue
                    wdir_stats = []
                    for s, e in zip(ds['start_time'].values, ds['end_time'].values):
                        sel = met_dataset[var].sel(time=slice(s, e))
                        mode_val = fast_mode_rounded(sel.to_series())
                        wdir_stats.append(mode_val)
                    ds[f'{var}_mode'] = (('event_id'), wdir_stats)

            # Add this instrument dataset to the list for this benchmark
            new_ds_list.append(ds)

        # Concatenate across instruments for this benchmark
        combined_instrument_ds = xr.concat(new_ds_list, dim="test_instrument")
        benchmark_ds_list.append(combined_instrument_ds)

    # Concatenate across benchmarks
    combined_event_ds = xr.concat(benchmark_ds_list, dim="benchmark")
    return combined_event_ds

## Identify Gothic Events

In [29]:
site = 'gothic'
# precipitation datasets
gothic_prcp_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/final/{site}_precipitation_30min.nc')
era5_land_prcp_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/ERA5-Land/era5_land_gothic_1hr.nc')
prism_prcp_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/PRISM/prism_site_data.nc').sel(site=site)
# meteorological datasets
gothic_met_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/SAIL/met_30min.nc')
gothic_bb_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/billy_barr/billy_barr_20211001-20230930_30min.nc')
# normalized meteorological datasets
gothic_met_norm_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/final/gothicMet_met_normalized.nc')
gothic_bb_norm_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/final/gothicBB_met_normalized.nc')

# add wind components to gothic_bb_ds
u, v = calculate_wind_components(gothic_bb_ds['windSpeed'], gothic_bb_ds['windDirec'])
gothic_bb_ds = gothic_bb_ds.assign({'u_wind': (('time'), u),
                                     'v_wind': (('time'), v)})

# rename variables for consistency
rename_dict_sail = {
        'atmos_pressure': 'pressure',
        'temp_mean': 'temperature',
        'rh_mean': 'relative_humidity',
        'wspd_vec_mean': 'wind_speed',
        'u': 'u_wind',
        'v': 'v_wind',
    }
gothic_met_ds = gothic_met_ds.rename(rename_dict_sail)

rename_dict_bb = {'baromPress':'pressure',
                  'avAirTemp':'temperature',
                  'relHumidty':'relative_humidity',
                  'windSpeed':'wind_speed'
}
gothic_bb_ds = gothic_bb_ds.rename(rename_dict_bb)

### Precipitation Observations

In [30]:
events_ds_list = []
benchmarks = ['billy_barr_precip', 'sail_pluvio']
for benchmark in benchmarks:
    for test_gauge in list(gothic_prcp_ds.data_vars):
        if test_gauge == benchmark:
            print("Skipping same gauge:", test_gauge)
            pass
        events_ds_list.append(find_events_union(gothic_prcp_ds,
                                      precip_benchmark=benchmark,
                                      precip_test=test_gauge,
                                      precip_thresh=PRECIP_THRESHOLD, 
                                      max_gap_steps=DRY_GAP_STEPS, 
                                      min_event_length=2,
                                      resample_interval=None, 
                                      limit=50,
                                      site=site,
                                      to_xarray=True))
# Combine all event datasets into one
combinedGothicEvents_ds = xr.merge(events_ds_list)

Skipping same gauge: billy_barr_precip
Skipping same gauge: sail_pluvio


In [8]:
# Add meteorological variables to the dataset
VARS_TO_ADD = [
    'pressure',
    'temperature',
    'relative_humidity',
    'u_wind',
    'v_wind',
    'wind_speed',
    'wdir_vec_mean',
]

In [108]:
combinedGothicEventsWithMet_ds = combine_events(combinedGothicEvents_ds, gothic_met_ds, VARS_TO_ADD)
combinedGothicEventsWithMetBB_ds = combine_events(combinedGothicEvents_ds, gothic_bb_ds, VARS_TO_ADD)

In [113]:
combinedGothicEventsWithMetNormalized_ds = combine_events(combinedGothicEvents_ds, gothic_met_norm_ds, VARS_TO_ADD)
combinedGothicEventsWithMetBBNormalized_ds = combine_events(combinedGothicEvents_ds, gothic_bb_norm_ds, VARS_TO_ADD)

### Gridded Precipitation Data

In [114]:
mergedGridded_ds = xr.merge([gothic_prcp_ds['billy_barr_precip'].resample(time='1d').sum(), 
                             gothic_prcp_ds['sail_pluvio'].resample(time='1d').sum(), 
                             era5_land_prcp_ds['tp'].resample(time='1d').sum(), 
                             prism_prcp_ds['ppt'].resample(time='1d').sum()])
# rename ppt to prism_pprt and tp to era5_land_tp
mergedGridded_ds = mergedGridded_ds.rename({'ppt':'prism_ppt', 'tp':'era5_land_tp'})

# set minimum threshold to 0.1 mm for all values
mergedGridded_ds = mergedGridded_ds.where((mergedGridded_ds > 1) | np.isnan(mergedGridded_ds), 0)

In [115]:
events_ds_list = []
benchmarks = ['billy_barr_precip', 'sail_pluvio']
for benchmark in benchmarks:
    for test_gauge in list(mergedGridded_ds.data_vars):
        if test_gauge == benchmark:
            print("Skipping same gauge:", test_gauge)
            continue
        events_ds_list.append(find_events_union(mergedGridded_ds,
                                      precip_benchmark=benchmark,
                                      precip_test=test_gauge,
                                      precip_thresh=PRECIP_THRESHOLD, 
                                      max_gap_steps=0, 
                                      resample_interval='1d', 
                                      limit=50,
                                      site=site,
                                      to_xarray=True))
# Combine all event datasets into one
combinedGothicGriddedEvents_ds = xr.merge(events_ds_list)

Skipping same gauge: billy_barr_precip


Skipping same gauge: sail_pluvio


In [116]:
### add event specific meteorology
# SAIL Met
gothic_met_df = gothic_met_ds.to_dataframe()
gothic_met_df = gothic_met_df.resample('1D').agg({'temperature':'mean',
                                                 'relative_humidity':'mean',
                                                 'vapor_pressure_mean':'mean',
                                                 'pressure':'mean',
                                                 'u_wind':'mean',
                                                 'v_wind':'mean',
                                                 'wind_speed':'mean',
                                                 'wdir_vec_mean':fast_mode_rounded,})
gothic_met_ds_daily = gothic_met_df.to_xarray()

# Billy Barr Met
gothic_bb_df = gothic_bb_ds.to_dataframe()
gothic_bb_df = gothic_bb_df.resample('1D').agg({'temperature':'mean',
                                               'relative_humidity':'mean',
                                               'pressure':'mean',
                                               'u_wind':'mean',
                                               'v_wind':'mean',
                                               'wind_speed':'mean',})
gothic_bb_ds_daily = gothic_bb_df.to_xarray()

# Normalized SAIL Met
gothic_met_norm_df = gothic_met_norm_ds.to_dataframe()
gothic_met_norm_df = gothic_met_norm_df.resample('1D').agg({'temperature':'mean',
                                                 'relative_humidity':'mean',
                                                 'pressure':'mean',
                                                 'u_wind':'mean',
                                                 'v_wind':'mean',
                                                 'wind_speed':'mean',})
gothic_met_norm_ds_daily = gothic_met_norm_df.to_xarray()

# Normalized Billy Barr Met
gothic_bb_norm_df = gothic_bb_norm_ds.to_dataframe()
gothic_bb_norm_df = gothic_bb_norm_df.resample('1D').agg({'temperature':'mean',
                                               'relative_humidity':'mean',
                                               'pressure':'mean',
                                               'u_wind':'mean',
                                               'v_wind':'mean',
                                               'wind_speed':'mean',})
gothic_bb_norm_ds_daily = gothic_bb_norm_df.to_xarray()                                                 

In [117]:
# Add meteorological variables to the dataset
combinedGothicGriddedEventsWithMet_ds = combine_events(combinedGothicGriddedEvents_ds, gothic_met_ds_daily, VARS_TO_ADD)
combinedGothicGriddedEventsWithMetBB_ds = combine_events(combinedGothicGriddedEvents_ds, gothic_bb_ds_daily, VARS_TO_ADD)
# Add normalized meteorological variables
combinedGothicGriddedEventsWithMetNormalized_ds = combine_events(combinedGothicGriddedEvents_ds, gothic_met_norm_ds_daily, VARS_TO_ADD)
combinedGothicGriddedEventsWithMetBBNormalized_ds = combine_events(combinedGothicGriddedEvents_ds, gothic_bb_norm_ds_daily, VARS_TO_ADD)

### Save Files

In [118]:
### Save the datasets
combinedGothicEvents_ds.to_netcdf(f'{OUTPUT_DIR}gothic_precipitation_event_comparisons.nc')
combinedGothicGriddedEvents_ds.to_netcdf(f'{OUTPUT_DIR}gothic_gridded_precipitation_event_comparisons.nc')

# SAIL Met
combinedGothicEventsWithMet_ds.to_netcdf(f'{OUTPUT_DIR}gothic_precipitation_event_comparisons_sail_with_raw_met.nc')
combinedGothicGriddedEventsWithMet_ds.to_netcdf(f'{OUTPUT_DIR}gothic_gridded_precipitation_event_comparisons_sail_with_raw_met.nc')

# Billy Barr Met
combinedGothicEventsWithMetBB_ds.to_netcdf(f'{OUTPUT_DIR}gothic_precipitation_event_comparisons_bb_with_raw_met.nc')
combinedGothicGriddedEventsWithMetBB_ds.to_netcdf(f'{OUTPUT_DIR}gothic_gridded_precipitation_event_comparisons_bb_with_raw_met.nc')

# Normalized meteorological datasets
# SAIL met
combinedGothicEventsWithMetNormalized_ds.to_netcdf(f'{OUTPUT_DIR}gothic_precipitation_event_comparisons_sail_with_normalized_met.nc')
combinedGothicGriddedEventsWithMetNormalized_ds.to_netcdf(f'{OUTPUT_DIR}gothic_gridded_precipitation_event_comparisons_sail_with_normalized_met.nc')

# Billy Barr met
combinedGothicEventsWithMetBBNormalized_ds.to_netcdf(f'{OUTPUT_DIR}gothic_precipitation_event_comparisons_bb_with_normalized_met.nc')
combinedGothicGriddedEventsWithMetBBNormalized_ds.to_netcdf(f'{OUTPUT_DIR}gothic_gridded_precipitation_event_comparisons_bb_with_normalized_met.nc')

## Identify Kettle Ponds Events

In [119]:
site = 'kettle_ponds'
kettle_ponds_prcp_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/final/{site}_precipitation_30min.nc')
era5_land_prcp_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/ERA5-Land/era5_land_gothic_1hr.nc')
prism_prcp_ds = xr.open_dataset('/storage/dlhogan/precipitation-rodeo/data/processed/final/PRISM/prism_site_data.nc').sel(site=site)
# meteorological datasets
kettle_ponds_asfs_met_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/SPLASH/asfs30_30min.nc')
kettle_ponds_sos_met_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/SOS/sos_ds_30min.nc')
# normalized meteorological datasets
kettle_ponds_asfs_met_norm_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/final/kettlePondsAsfs30_met_normalized.nc')
kettle_ponds_sos_met_norm_ds = xr.open_dataset(f'/storage/dlhogan/precipitation-rodeo/data/processed/final/kettlePondsSOS_met_normalized.nc')
# rename variables for consistency
rename_dict_sos = {'P_10m_c': 'pressure',
                   'T_3m_c': 'temperature',
                   'RH_3m_c': 'relative_humidity',
                   'spd_10m_c': 'wind_speed',
                   'dir_10m_c': 'wind_direction',
                   'u_10m_c': 'u_wind',
                   'v_10m_c': 'v_wind'}
kettle_ponds_sos_met_ds = kettle_ponds_sos_met_ds.rename(rename_dict_sos)

rename_dict_asfs = {'atmos_pressure': 'pressure',
                    'temp': 'temperature',
                    'rh': 'relative_humidity',
                    'wspd_u_mean': 'u_wind',
                    'wspd_v_mean': 'v_wind',
                    }
kettle_ponds_asfs_met_ds = kettle_ponds_asfs_met_ds.rename(rename_dict_asfs)

# calculate wind speed
kettle_ponds_asfs_met_ds['wind_speed'] = np.sqrt(kettle_ponds_asfs_met_ds['u_wind']**2 + kettle_ponds_asfs_met_ds['v_wind']**2)

### Precipitation Observations

In [120]:
events_ds_list = []
benchmarks = ['billy_barr_precip', 'splash_lpdf']
for benchmark in benchmarks:
    for test_gauge in list(kettle_ponds_prcp_ds.data_vars):
        if test_gauge == benchmark:
            print("Skipping same gauge:", test_gauge)
            pass
        events_ds_list.append(find_events_union(kettle_ponds_prcp_ds,
                                      precip_benchmark=benchmark,
                                      precip_test=test_gauge,
                                      precip_thresh=PRECIP_THRESHOLD, 
                                      max_gap_steps=DRY_GAP_STEPS, 
                                      min_event_length=2,
                                      resample_interval=None, 
                                      limit=50,
                                      site=site,
                                      to_xarray=True))
# Combine all event datasets into one
combinedKettlePondsEvents_ds = xr.merge(events_ds_list)

Skipping same gauge: billy_barr_precip
Skipping same gauge: splash_lpdf


In [121]:
combinedKettlePondsEventsWithMetASFS_ds = combine_events(combinedKettlePondsEvents_ds, kettle_ponds_asfs_met_ds, VARS_TO_ADD)
combinedKettlePondsEventsWithMetSOS_ds = combine_events(combinedKettlePondsEvents_ds, kettle_ponds_sos_met_ds, VARS_TO_ADD)
combinedKettlePondsEventsWithMetNormalizedASFS_ds = combine_events(combinedKettlePondsEvents_ds, kettle_ponds_asfs_met_norm_ds, VARS_TO_ADD)
combinedKettlePondsEventsWithMetBBNormalizedSOS_ds = combine_events(combinedKettlePondsEvents_ds, kettle_ponds_sos_met_norm_ds, VARS_TO_ADD)

### Gridded Precipitation Data

In [122]:
mergedGridded_ds = xr.merge([kettle_ponds_prcp_ds['billy_barr_precip'].resample(time='1d').sum(), 
                             kettle_ponds_prcp_ds['splash_lpdf'].resample(time='1d').sum(), 
                             era5_land_prcp_ds['tp'].resample(time='1d').sum(), 
                             prism_prcp_ds['ppt'].resample(time='1d').sum()])
# rename ppt to prism_pprt and tp to era5_land_tp
mergedGridded_ds = mergedGridded_ds.rename({'ppt':'prism_ppt', 'tp':'era5_land_tp'})

# set minimum threshold to 0.1 mm for all values
mergedGridded_ds = mergedGridded_ds.where((mergedGridded_ds > 1) | np.isnan(mergedGridded_ds), 0)

In [123]:
events_ds_list = []
benchmarks = ['billy_barr_precip', 'splash_lpdf']
for benchmark in benchmarks:
    for test_gauge in list(mergedGridded_ds.data_vars):
        if test_gauge == benchmark:
            print("Skipping same gauge:", test_gauge)
            continue
        events_ds_list.append(find_events_union(mergedGridded_ds,
                                      precip_benchmark=benchmark,
                                      precip_test=test_gauge,
                                      precip_thresh=PRECIP_THRESHOLD, 
                                      max_gap_steps=0, 
                                      resample_interval='1d', 
                                      limit=50,
                                      site=site,
                                      to_xarray=True))
# Combine all event datasets into one
combinedKettlePondsGriddedEvents_ds = xr.merge(events_ds_list)

Skipping same gauge: billy_barr_precip
Skipping same gauge: splash_lpdf


In [124]:
### add event specific meteorology
# SOS Met
kettle_ponds_sos_met_df = kettle_ponds_sos_met_ds.to_dataframe()
kettle_ponds_sos_met_df = kettle_ponds_sos_met_df.resample('1D').agg({'temperature':'mean',
                                                 'relative_humidity':'mean',
                                                 'pressure':'mean',
                                                 'u_wind':'mean',
                                                 'v_wind':'mean',
                                                 'wind_speed':'mean'})
kettle_ponds_sos_met_ds_daily = kettle_ponds_sos_met_df.to_xarray()

# ASFS Met
kettle_ponds_asfs_met_df = kettle_ponds_asfs_met_ds.to_dataframe()
kettle_ponds_asfs_met_df = kettle_ponds_asfs_met_df.resample('1D').agg({'temperature':'mean',
                                               'relative_humidity':'mean',
                                               'pressure':'mean',
                                               'u_wind':'mean',
                                               'v_wind':'mean',
                                               'wind_speed':'mean',})
kettle_ponds_asfs_met_ds_daily = kettle_ponds_asfs_met_df.to_xarray()

# Normalized SOS Met
kettle_ponds_sos_met_norm_df = kettle_ponds_sos_met_norm_ds.to_dataframe()
kettle_ponds_sos_met_norm_df = kettle_ponds_sos_met_norm_df.resample('1D').agg({'temperature':'mean',
                                                 'relative_humidity':'mean',
                                                 'pressure':'mean',
                                                 'u_wind':'mean',
                                                 'v_wind':'mean',
                                                 'wind_speed':'mean',})
kettle_ponds_sos_met_norm_ds_daily = kettle_ponds_sos_met_norm_df.to_xarray()

# Normalized ASFS Met
kettle_ponds_asfs_met_norm_df = kettle_ponds_asfs_met_norm_ds.to_dataframe()
kettle_ponds_asfs_met_norm_df = kettle_ponds_asfs_met_norm_df.resample('1D').agg({'temperature':'mean',
                                               'relative_humidity':'mean',
                                               'pressure':'mean',
                                               'u_wind':'mean',
                                               'v_wind':'mean',
                                               'wind_speed':'mean',})
kettle_ponds_asfs_met_norm_ds_daily = kettle_ponds_asfs_met_norm_df.to_xarray()

In [125]:
combinedKettlePondsGriddedEventsWithMetASFS_ds = combine_events(combinedKettlePondsGriddedEvents_ds, kettle_ponds_asfs_met_ds_daily, VARS_TO_ADD)
combinedKettlePondsGriddedEventsWithMetSOS_ds = combine_events(combinedKettlePondsGriddedEvents_ds, kettle_ponds_sos_met_ds_daily, VARS_TO_ADD)
# Add normalized meteorological variables
combinedKettlePondsGriddedEventsWithMetNormalizedASFS_ds = combine_events(combinedKettlePondsGriddedEvents_ds, kettle_ponds_asfs_met_norm_ds_daily, VARS_TO_ADD)
combinedKettlePondsGriddedEventsWithMetBBNormalizedSOS_ds = combine_events(combinedKettlePondsGriddedEvents_ds, kettle_ponds_sos_met_norm_ds_daily, VARS_TO_ADD)

In [126]:
# Save the datasets
combinedKettlePondsEvents_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_precipitation_event_comparisons.nc')
combinedKettlePondsGriddedEvents_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_gridded_precipitation_event_comparisons.nc')
# ASFS datasets
combinedKettlePondsEventsWithMetASFS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_precipitation_event_comparisons_asfs_with_raw_met.nc')
combinedKettlePondsGriddedEventsWithMetASFS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_gridded_precipitation_event_comparisons_asfs_with_raw_met.nc')

# SOS datasets
combinedKettlePondsEventsWithMetSOS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_precipitation_event_comparisons_sos_with_raw_met.nc')
combinedKettlePondsGriddedEventsWithMetSOS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_gridded_precipitation_event_comparisons_sos_with_raw_met.nc')

# Normalized meteorological datasets
# ASFS datasets
combinedKettlePondsEventsWithMetNormalizedASFS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_precipitation_event_comparisons_asfs_with_normalized_met.nc')
combinedKettlePondsGriddedEventsWithMetNormalizedASFS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_gridded_precipitation_event_comparisons_asfs_with_normalized_met.nc')

# SOS datasets
combinedKettlePondsEventsWithMetBBNormalizedSOS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_precipitation_event_comparisons_sos_with_normalized_met.nc')
combinedKettlePondsGriddedEventsWithMetBBNormalizedSOS_ds.to_netcdf(f'{OUTPUT_DIR}/kettle_ponds_gridded_precipitation_event_comparisons_sos_with_normalized_met.nc')

## Sanity Check

In [115]:
for test_gauge in combinedKettlePondsEvents_ds.test_instrument.values: 
    print(test_gauge)
    print(combinedKettlePondsEvents_ds.sel(benchmark='splash_lpdf', test_instrument=test_gauge)['abs_diff'].mean().values)

billy_barr_precip
8.36032
rain_rate_A
23.295208208909308
sail_squire_rain
23.639624506519336
sail_squire_snow_m2009_1
19.88312126825882
sail_squire_snow_m2009_2
21.167994815703135
sail_squire_snow_ws2012
19.523436127015426
sail_squire_snow_ws88diw
16.156996103141932
snow_rate_m2009_1
15.715477238536678
snow_rate_m2009_2
17.77404070862194
snow_rate_ws2012
14.714995035942461
snow_rate_ws88diw
15.633594037107633
sos_swe_p1
17.91904499816895
sos_swe_p2
18.727096286010745
sos_swe_p3
18.69401524047852
sos_swe_p4
17.94047116851807
splash_ld_uncorrected
13.745200000000004
splash_lpdf
0.0
